<a href="https://colab.research.google.com/github/sleepymor/image-recognition-learning/blob/main/Image_Preprocessing_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Image Dataset Preprocessing for PyTorch

This section will cover the essential steps to preprocess a raw image dataset for training a computer vision model in PyTorch. We'll define image transformations, load the dataset, and prepare it for training using a `DataLoader`.

In [ ]:
# Install torchvision if not already installed
# !pip install torchvision

import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os


#### 1. Define Transformations

We'll define a series of transformations to apply to our images. These typically include resizing, cropping, converting to a PyTorch tensor, and normalizing the pixel values. The normalization parameters (mean and standard deviation) are usually based on a large dataset like ImageNet, or can be calculated from your specific dataset.

In [ ]:
# Define your image transformations
# These are common transformations for training a CNN
# You might need to adjust them based on your dataset and model requirements
transform = transforms.Compose([
    transforms.Resize((224, 224)), # Resize all images to 224x224 pixels
    transforms.RandomHorizontalFlip(), # Randomly flip the image horizontally for data augmentation
    transforms.ToTensor(), # Convert the image to a PyTorch tensor
    # Normalize the image: (pixel - mean) / std
    # These values are common for ImageNet-trained models, adjust if needed
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# You might want different transformations for validation/testing (e.g., no random flip)
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


#### 2. Load the Dataset

`torchvision.datasets.ImageFolder` is a convenient way to load image data organized in a standard directory structure (e.g., `root/class_name/image.jpg`). Make sure your `data_dir` points to the root directory containing your class subfolders.

In [ ]:
# Specify the path to your dataset
# Assuming your dataset is organized as: data_dir/class1/img1.jpg, data_dir/class2/img2.jpg, etc.
# Use the text input below to provide your dataset path (e.g., a path to a folder on Google Drive or a local directory)
data_dir = "/content/drive/MyDrive/my_image_dataset" #@param {type:"string"}

# !!! IMPORTANT !!!
# Replace the default path above with the actual path to your dataset.
# For example, if your images are in a folder named 'my_images' in your Google Drive,
# you would set data_dir = '/content/drive/MyDrive/my_images'


try:
    dataset = ImageFolder(root=data_dir, transform=transform)
    print(f"Successfully loaded dataset with {len(dataset)} images.")
    print(f"Classes: {dataset.classes}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Please ensure the `data_dir` path is correct and contains subfolders for each class.")
    print("Example structure: `data_dir/class_name_1/img1.jpg`, `data_dir/class_name_2/img2.png`")


#### 3. Create DataLoaders

A `DataLoader` helps iterate over the dataset in batches, shuffle the data, and load it in parallel using multiple worker processes. This is crucial for efficient training.

In [ ]:
# Define batch size
batch_size = 32

# Create a DataLoader for the training set
# num_workers > 0 can speed up data loading, especially with many transformations
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

print(f"Training DataLoader created with batch size {batch_size}.")

# Example: Get one batch of images and labels
if dataset:
    for images, labels in train_loader:
        print(f"Shape of image batch: {images.shape}") # Should be [batch_size, channels, height, width]
        print(f"Shape of label batch: {labels.shape}") # Should be [batch_size]
        break

# If you have a separate validation/test set, you'd load it similarly:
# test_dataset = ImageFolder(root='path/to/test/dataset', transform=transform_test)
# test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


#### 4. Visualize a Sample Batch (Optional)

It's often helpful to visualize some images after transformations to ensure everything is set up correctly, especially normalization. Note that `matplotlib` expects image arrays in `(height, width, channels)` format, and `ToTensor()` converts to `(channels, height, width)`. Also, normalization will make pixel values fall outside the 0-1 range, so we'll need to reverse it for proper display.

In [ ]:
if dataset:
    # Function to un-normalize and show an image
    def imshow(img):
        # Un-normalize
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = img.numpy().transpose((1, 2, 0)) # Convert from (C, H, W) to (H, W, C)
        img = std * img + mean # Unnormalize
        img = np.clip(img, 0, 1)
        plt.imshow(img)
        plt.show()

    # Get a batch of training data
    dataiter = iter(train_loader)
    images, labels = next(dataiter)

    # Show images
    fig = plt.figure(figsize=(10, 8))
    for i in range(min(4, len(images))):
        ax = fig.add_subplot(1, 4, i+1, xticks=[], yticks=[])
        imshow(images[i])
        if dataset.classes:
            ax.set_title(dataset.classes[labels[i]])
        else:
            ax.set_title(f"Label: {labels[i].item()}")
    plt.tight_layout()
    plt.show()


#### 5. Visualize a Single Processed Image

Let's pick one image from the dataset, apply the transformations, and display it to see the effect of the preprocessing, including resizing, flipping (if randomly applied), and normalization (which is reversed for display).

In [ ]:
if dataset:
    # Define imshow function again for clarity, or ensure cell 8a880052 was run
    def imshow(img):
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = img.numpy().transpose((1, 2, 0))
        img = std * img + mean
        img = np.clip(img, 0, 1)
        plt.imshow(img)
        plt.show()

    # Get a single image and its label from the dataset
    # We'll take the first image for demonstration
    sample_image, sample_label = dataset[0]

    print(f"Original image shape (after ToTensor and Normalize): {sample_image.shape}")
    if dataset.classes:
        print(f"Image belongs to class: {dataset.classes[sample_label]}")
    else:
        print(f"Image belongs to label: {sample_label}")

    plt.figure(figsize=(6, 6))
    imshow(sample_image)
    plt.title(f"Processed Image (Class: {dataset.classes[sample_label] if dataset.classes else sample_label})")
    plt.axis('off')
    plt.show()
